In [1]:
from IPython.display import display

from eki_dev.aws_service import AwsService
from aws_cluster import pest_cluster
from aws_cluster.cluster_utils import (check_resource_creation_status,
check_stack_creation_status)



# EKI PEST_HP Cluster Dashboard

## Path to Yaml Configuration

In [2]:
path_yaml_config = 's3://scratch-marco/parameters.yaml'

## STEP 1: Create Cluster

In [3]:
if pest_cluster.create_pest_cluster_stack(path_yaml_config):
    print('Pest cluster stack creation initiated')

Pest cluster stack creation initiated


## Monitor Cluster Creation and Status

Use the cell below to monitor the status of the different elastic components that form the cluster. Initially the service is created with a single agent waiting for work from the main task that holds the host procses (created below). The 

In [11]:

cluster_status = check_stack_creation_status("PestClusterInfrastructure")
print(f"The current status of the cluster formation is {cluster_status}")
print("Do not proceed until the cluster formation is complete.")
display(check_resource_creation_status("PestClusterInfrastructure"))

The current status of the cluster formation is DELETE_IN_PROGRESS
Do not proceed until the cluster formation is complete.


KeyError: 'DELETE_SKIPPED'

## STEP 2: Create main task

Creates the main task in the cluster running the PEST host process. The agents in the cluster will start running model instances as soon as the task is created. 

In [4]:
res = pest_cluster.create_main_task(path_yaml_config)

Waiting for Main Task to be Created...
Main Task Created. Registering IP in Target Group
Main Instance IP: 10.10.28.195


In [6]:
desired_number_agents = 2


### Do not modify anything below this line ######
pest_cluster.update_number_agents(path_yaml_config, desired_number_agents)


{'service': {'serviceArn': 'arn:aws:ecs:us-west-1:054507568115:service/PestCluster/agent_private_net',
  'serviceName': 'agent_private_net',
  'clusterArn': 'arn:aws:ecs:us-west-1:054507568115:cluster/PestCluster',
  'loadBalancers': [],
  'serviceRegistries': [],
  'status': 'ACTIVE',
  'desiredCount': 2,
  'runningCount': 1,
  'pendingCount': 0,
  'capacityProviderStrategy': [{'capacityProvider': 'autoscaling-private-subnet-for-ecs',
    'weight': 1,
    'base': 0}],
  'taskDefinition': 'arn:aws:ecs:us-west-1:054507568115:task-definition/pest_agent:15',
  'deploymentConfiguration': {'deploymentCircuitBreaker': {'enable': True,
    'rollback': True},
   'maximumPercent': 200,
   'minimumHealthyPercent': 100,
   'alarms': {'alarmNames': [], 'enable': False, 'rollback': False}},
  'deployments': [{'id': 'ecs-svc/5282727034476831068',
    'status': 'PRIMARY',
    'taskDefinition': 'arn:aws:ecs:us-west-1:054507568115:task-definition/pest_agent:15',
    'desiredCount': 1,
    'pendingCount

In [22]:
# cf = AwsService.from_service('cloudformation')
# stack_resources = cf.client.describe_stack_resources(StackName="PestClusterInfrastructure")
# for resource in stack_resources['StackResources']:
#     if resource['ResourceType'] == "AWS::ECS::Service":
#         service_arn = resource['PhysicalResourceId']
#         print(service_arn)
        

arn:aws:ecs:us-west-1:054507568115:service/PestCluster/agent_private_net


In [5]:
pest_cluster.list_agent_tasks(path_yaml_config)


,0
0,arn:aws:ecs:us-west-1:054507568115:task/PestCl...


In [ ]:
logs_client = AwsService.from_service("logs").client
 
response = logs_client.get_log_events(
        logGroupName="/ecs/pest_agent",
        logStreamName="ecs/model/f7389a4dc2374efb91e3a2be853b92a8",
        limit=100,
        startFromHead=False
)
for event in response["events"]:
        print(event["timestamp"], event["message"])

In [16]:
# ecs = AwsService.from_service('ecs')
# service_arns = ecs.client.list_services(cluster="PestCluster")["serviceArns"]
# print(service_arns)
# ecs.client.list_tasks(
#         cluster="PestCluster",
#         serviceName=service_arns[0]
#     )

#ecs.client.describe_task_sets(cluster="PestCluster", service=service_arn)

In [12]:
main_task = cf.client.describe_tasks(cluster="PestCluster", tasks=['arn:aws:ecs:us-west-1:054507568115:task/PestCluster/f65a622afc4e434dbfdaef14d7a550bb'])


In [18]:
main_task

{'tasks': [{'attachments': [],
   'attributes': [{'name': 'ecs.cpu-architecture', 'value': 'x86_64'}],
   'availabilityZone': 'us-west-1c',
   'capacityProviderName': 'Infra-ECS-Cluster-PestCluster-a8643df8-EC2CapacityProvider-HxlrJRzZVZ1l',
   'clusterArn': 'arn:aws:ecs:us-west-1:054507568115:cluster/PestCluster',
   'connectivity': 'CONNECTED',
   'connectivityAt': datetime.datetime(2024, 10, 13, 18, 10, 16, 300000, tzinfo=tzlocal()),
   'containerInstanceArn': 'arn:aws:ecs:us-west-1:054507568115:container-instance/PestCluster/45479f3eeec740959e36a01f125834d1',
   'containers': [{'containerArn': 'arn:aws:ecs:us-west-1:054507568115:container/PestCluster/f65a622afc4e434dbfdaef14d7a550bb/13ec9e27-8ac3-4449-baf4-428dd4a682e5',
     'taskArn': 'arn:aws:ecs:us-west-1:054507568115:task/PestCluster/f65a622afc4e434dbfdaef14d7a550bb',
     'name': 'model',
     'image': '054507568115.dkr.ecr.us-west-1.amazonaws.com/ww_2024:dev',
     'imageDigest': 'sha256:0198069cc2cb5d2b7eb95c461ffcdbbc81213

In [16]:
main_task['tasks'][0]['containerInstanceArn']

'arn:aws:ecs:us-west-1:054507568115:container-instance/PestCluster/45479f3eeec740959e36a01f125834d1'

In [20]:
main_container = cf.client.describe_container_instances(cluster="PestCluster", 
                                       containerInstances=[main_task['tasks'][0]['containerInstanceArn']])

In [23]:
main_instance = main_container['containerInstances'][0]['ec2InstanceId']

In [22]:
cf = AwsService.from_service('ec2')

In [26]:
main_instance_ip = cf.client.describe_instances(InstanceIds = [main_instance])

In [32]:
main_instance_ip = main_instance_ip['Reservations'][0]['Instances'][0]['PrivateIpAddress']

In [36]:
cf = AwsService.from_service('ecs')

'arn:aws:elasticloadbalancing:us-west-1:054507568115:targetgroup/PestMain/989eb9bf9ee12230'

In [46]:
cf = AwsService.from_service('elbv2')
cf.client.register_targets(TargetGroupArn=target_group_arn,
                           Targets=[{
                               'Id': main_instance_ip,
                               'Port': 4004,
                           }])

{'ResponseMetadata': {'RequestId': '8633782f-f43d-4851-b8cd-832c476333e3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '8633782f-f43d-4851-b8cd-832c476333e3',
   'content-type': 'text/xml',
   'content-length': '253',
   'date': 'Mon, 14 Oct 2024 02:19:12 GMT'},
  'RetryAttempts': 0}}

In [23]:
import pandas as pd
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

In [76]:
df = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df

,A,B
0,1,3
1,2,4


In [82]:
    
df_style = df.style.apply(lambda x: ['background-color: green'], axis=0)
    

In [84]:
df_style

,A,B
0,1,3
1,2,4


In [6]:
pest_cluster.terminate_cluster()

{'ResponseMetadata': {'RequestId': '895277ef-3266-483b-9bac-f11a7ef87a49',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '895277ef-3266-483b-9bac-f11a7ef87a49',
   'date': 'Mon, 28 Oct 2024 08:22:34 GMT',
   'content-type': 'text/xml',
   'content-length': '212',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}